In [6]:
import json
import pyvo
import astropy.coordinates
import astropy.units as u
import numpy as np

from astroquery.alma import Alma
from astropy.time import Time
from collections import defaultdict


In [7]:
alma = Alma()   

In [8]:
# Resolve position
coord = astropy.coordinates.SkyCoord.from_name("HH 212")

BAND_FILTER = ["6", "7"]
SEARCH_RADIUS_ARCSEC = 60
MAX_RESOLUTION_ARCSEC = 1

tbl = alma.query_region(
    coord,
    radius=SEARCH_RADIUS_ARCSEC * u.arcsec,
    band_list=BAND_FILTER,
    public=True,
    science=True,
)

# Angular resolution filtering
tbl = tbl[tbl["spatial_resolution"] < MAX_RESOLUTION_ARCSEC] # Note: It looks like the spatial_resolution column is in arcseconds / is actually the anngular resolution

KeyboardInterrupt: 

NOTE on combined continuum images: 

It looks like ivoa.obscore rows are per-spectral-window metadata records. The combined continuum FITS product is a pipeline-generated file that combines several of those SPW rows into one derived product, and it does NOT have its own row in ivoa.obscore.

There's no obs_id to search for that will ever equal that combined-continuum filename, because ObsCore operates one level below it in the data hierarchy. This means I can't use "dataproduct_type" or "obs_id" to obtain the continuum images.

HOWEVER, there is a "cont_sensitivity_bandwidth" column in the ObsCore schema. It looks like a populated/non-null "cont_sensitivity_bandwidth" on a set of SPW rows signals that continuum imaging was likely performed using those windows.


In [9]:
for row in tbl:
    print(row['cont_sensitivity_bandwidth'], row['proposal_id'], row['obs_id'], row['frequency'], row['frequency_support'])

0.48429030348921565 2017.1.01053.S uid://A001/X1288/Xf93.source.HH_212.spw.16 360.0092825359129 [345.22..347.22GHz,1128.91kHz,11.8mJy/beam@10km/s,897.1uJy/beam@native, XX YY] U [347.09..349.09GHz,1128.91kHz,10.9mJy/beam@10km/s,831.2uJy/beam@native, XX YY] U [357.13..359.13GHz,1128.91kHz,14mJy/beam@10km/s,1.1mJy/beam@native, XX YY] U [359.01..361.01GHz,1128.91kHz,15.2mJy/beam@10km/s,1.2mJy/beam@native, XX YY]
0.48429030348921565 2017.1.01053.S uid://A001/X1288/Xf93.source.HH_212.spw.18 358.1342323015657 [345.22..347.22GHz,1128.91kHz,11.8mJy/beam@10km/s,897.1uJy/beam@native, XX YY] U [347.09..349.09GHz,1128.91kHz,10.9mJy/beam@10km/s,831.2uJy/beam@native, XX YY] U [357.13..359.13GHz,1128.91kHz,14mJy/beam@10km/s,1.1mJy/beam@native, XX YY] U [359.01..361.01GHz,1128.91kHz,15.2mJy/beam@10km/s,1.2mJy/beam@native, XX YY]
0.48429030348921565 2017.1.01053.S uid://A001/X1288/Xf93.source.HH_212.spw.20 348.0919670745972 [345.22..347.22GHz,1128.91kHz,11.8mJy/beam@10km/s,897.1uJy/beam@native, XX YY] U

In [9]:
tbl = alma.query_region(
    coord,
    radius=SEARCH_RADIUS_ARCSEC * u.arcsec,
    band_list=BAND_FILTER,
    public=True,
    science=True,
)

# Filter by spatial resolution (in arcseconds, so actually an angular resolution)
tbl = tbl[tbl["spatial_resolution"] < MAX_RESOLUTION_ARCSEC]

# NOTE: ObsCore/TAP returns one row per spectral window, so a single Member OUS
# (MOUS) can appear as several rows. Group by member_ous_uid so each
# physical observation is reported exactly once.
groups = defaultdict(list)
for row in tbl:
    groups[str(row["member_ous_uid"])].append(row)

In [11]:
for mous_uid, rows in groups.items():
    print(mous_uid, rows)
    print()

uid://A001/X1284/X17ba [<Row index=0>
pol_states      obs_publisher_did      obs_collection facility_name instrument_name                    obs_id                   dataproduct_type calib_level collections target_name        s_ra              s_dec               s_fov                         s_region                 s_xel1 s_xel2 em_xel t_xel pol_xel     s_resolution       t_min        t_max     t_exptime t_resolution         em_min                em_max           em_res_power    sensitivity_10kms  cont_sensitivity_bandwidth    pwv        group_ous_uid          member_ous_uid             asdm_uid                                             obs_title                                      type scan_intent science_observation spatial_scale_max  qa2_passed                                  bib_reference                                        science_keyword         scientific_category   pi_userid    pi_name    spectral_resolution       lastModified                     o_ucd                 

In [ ]:
import json
from collections import defaultdict

import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.alma import Alma

SEARCH_RADIUS_ARCSEC = 60.0
BAND_FILTER = ["6", "7"]
MAX_RESOLUTION_ARCSEC = 1.0

alma = Alma()

# Resolve position to avoid issues with target name variations (e.g., HH 212 vs. HH212)
coord = SkyCoord.from_name("HH 212")

#Perform query for public observations using the coordinates and search radius
tbl = alma.query_region(
    coord,
    radius=SEARCH_RADIUS_ARCSEC * u.arcsec,
    band_list=BAND_FILTER,
    public=True,
    science=True,
)

# Filter by spatial resolution (in arcseconds, so actually an angular resolution)
tbl = tbl[tbl["spatial_resolution"] < MAX_RESOLUTION_ARCSEC]

# NOTE: ObsCore/TAP returns one row per spectral window, so a single Member OUS
# (MOUS) can appear as several rows. Group by member_ous_uid so each
# physical observation is reported exactly once.
groups = defaultdict(list)
for row in tbl:
    groups[str(row["member_ous_uid"])].append(row)

observations = []
for mous_uid, rows in groups.items():
    bands = sorted({int(b) for r in rows for b in str(r["band_list"]).split()})
    resolutions = [float(r["spatial_resolution"]) for r in rows]
    observations.append({
        "member_ous_uid": mous_uid,
        "project_code": rows[0]["proposal_id"],
        "alma_source_name": rows[0]["target_name"],
        "ra_deg": float(rows[0]["s_ra"]),
        "dec_deg": float(rows[0]["s_dec"]),
        "band": bands,
        "spatial_resolution_arcsec": min(resolutions),
        "qa2_status": str(rows[0]["qa2_passed"]),
    })

all_bands = sorted({b for obs in observations for b in obs["band"]})
all_resolutions = [obs["spatial_resolution_arcsec"] for obs in observations]

report = {
    "target_name": "HH 212",
    "resolved_ra_deg": coord.ra.deg,
    "resolved_dec_deg": coord.dec.deg,
    "search_radius_arcsec": SEARCH_RADIUS_ARCSEC,
    "band_filter": [6, 7],
    "max_spatial_resolution_arcsec": MAX_RESOLUTION_ARCSEC,
    "num_observations": len(observations),
    "observations": observations,
    "summary": (
        f"HH 212 has {len(observations)} public ALMA Band 6/7 science "
        f"observations (grouped by Member OUS) with sub-arcsecond spatial "
        f"resolution, covering bands {all_bands}, with resolutions ranging "
        f"from {min(all_resolutions):.3f} to {max(all_resolutions):.3f} arcsec "
        f"across {len(np.unique([o['project_code'] for o in observations]))} "
        f"distinct projects."
    ) if observations else (
        "No public ALMA Band 6/7 science observations with sub-arcsecond "
        "spatial resolution were found for HH 212."
    ),
}

In [13]:
report

{'target_name': 'HH 212',
 'resolved_ra_deg': np.float64(85.96420833),
 'resolved_dec_deg': np.float64(-1.04808333),
 'search_radius_arcsec': 60.0,
 'band_filter': [6, 7],
 'max_spatial_resolution_arcsec': 1.0,
 'num_observations': 14,
 'observations': [{'member_ous_uid': 'uid://A001/X1284/X17ba',
   'project_code': np.str_('2017.1.00712.S'),
   'alma_source_name': np.str_('HH_212'),
   'ra_deg': 85.96421125004338,
   'dec_deg': -1.0481019444447166,
   'band': [6],
   'spatial_resolution_arcsec': 0.14893816480549288,
   'qa2_status': 'T'},
  {'member_ous_uid': 'uid://A001/X1284/X17b8',
   'project_code': np.str_('2017.1.00712.S'),
   'alma_source_name': np.str_('HH_212'),
   'ra_deg': 85.96421125001223,
   'dec_deg': -1.0481019444444382,
   'band': [6],
   'spatial_resolution_arcsec': 0.02278124950467702,
   'qa2_status': 'T'},
  {'member_ous_uid': 'uid://A001/X88f/X25b',
   'project_code': np.str_('2016.1.01475.S'),
   'alma_source_name': np.str_('HH212'),
   'ra_deg': 85.964208333384